<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/SFDS/egg_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Royal Draconic Conservatory**

<div style="text-align: center;">
   <img src="https://raw.githubusercontent.com/kwanda2426/projects/main/SFDS/egg.png"  width="800">
</div>




# **Importing Libraries**

In [43]:
!pip -q install plotly scikit-learn tabulate pandas xgboost

In [ ]:
# Libraries for data loading, data manipulation and data visulisation
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
import plotly.express as px
from scipy.stats import kurtosis


#Feature engineering, selection and Model training
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

#ignoring warnings
import warnings
warnings.filterwarnings('ignore')

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
from tabulate import tabulate
from IPython.display import display


# **Data acquisition**

#### Main Data

In [ ]:
# Loading Data
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/SFDS/egg_data.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
# Create the data as a list of dictionaries
variables = [

    {"variable": "MASS", "description": "Mass of the egg (kg)."},
    {"variable": "VOL", "description": "Volume of the egg (litres)."},
    {"variable": "AGE", "description": "Age of the egg (days since laying)."},
    {"variable": "NEXT", "description": "Type of nest (categorical; Volcanic, Forest, Mountain, Coastal or Swamp)."},
    {"variable": "SPOT", "description": "Number of visible shell spots"},
    {"variable": "TEMP", "description": " Surface Temprature of the egg in centigrade."},
    {"variable": "SPEC", "description": " Target variable (Dragon, Wyvern, Hydra)."},


]


# Display the DataFrame
var_desc = pd.DataFrame(variables)
var_desc_final = var_desc.reset_index(drop = True)

# Display the DataFrame
print(tabulate(var_desc_final, headers = 'keys', tablefmt = 'grid'))

# **Data Exploratory Analysis**

Data preprocessing is vital because it transforms messy raw data into a clean, consistent, and structured format, which is essential for accurate analysis and effective machine learning models.
Issues like missing values, inconsistencies, errors, and outliers are addressed to improve data quality,leads to more reliable and meaningful insights for decision-making and ultimately enhances the performance and accuracy of models.



In [ ]:
df_main = df.copy()

#### Reviewing and Renaming Columns


In [ ]:
# data columns
df_main.columns

- There are column names that have UPPERCASE. All variable names are converted to lowercase.

In [ ]:
# Changing column names
df_main.columns = df_main.columns.str.replace(' ', '_', regex=False).str.lower()
df_main.head()

#### Dataset Dimensions and Variable Type

In [ ]:
df = df_main.copy()

In [ ]:
# Rows and Columns of data
print('Data has {} rows and {} Columns'.format(df.shape[0],df.shape[1]))
print('')
print('Data has the following variables with missing values:')
print('')
major_missing = []
for i in df_main.columns:
  missing_values = 100*df_main[i].isnull().sum()/df_main.shape[0]
  print(i+', with {}% of missing data'.format(missing_values))


- No missing data.

#### SPEC - Target Variable Distribution

In [ ]:
df_main.spec.hist()

In [ ]:
# Number of observations
df_main.spec.value_counts()

**Observations**
- Hydra is the dominant class
Over half of the dataset belongs to Hydra.
This suggests the dataset is biased toward Hydra observations.
- Dragon is underrepresented compared to Wyvern and Hydra.

This could create a class imbalance problem in modelling implying that the model may learn Hydra and Wyvern  patterns better than Dragon patterns.
Metrics like accuracy alone may become misleading. Hence stratification is needed when splitting train/test data.

 #### NEST-  Type of nest Distribution

In [ ]:
df_main.nest.hist()

In [ ]:
# Number of observations
df_main.nest.value_counts()

**Observations**
- There's an issue with Swampp - it is a data issue which was supposed to be Swampp. This can be dropped because it is one datapoint.
- The counts across Mountain, Swamp, Forest, Volcanic, and Coastal nests are relatively close, meaning no single category heavily dominates the dataset.
- Mountain nests are the most common while Coastal nests are the least common
Mountain has the highest number of observations (116), whereas Coastal has the lowest (83), noting that the difference is not extreme.
- The feature is suitable for machine learning models
Because the categories are reasonably balanced, the feature can provide useful predictive information without causing major class imbalance issues.

#### Summary Statistics

In [ ]:
#describing numerical columns
df_main.describe().T

**Observations**:

- Both vol and mass appear heavily right-skewed. The means are far above the medians, which usually indicates potential outliers. This means that the data might need to be transformed or create a new feature from the two.

- mass has a negative minimum value. This is strange because a physical mass should not be negative. This might be because of calibration errors or data corruption. These data points can be removed or treated as anomalies.

- age looks uniformly distributed.

- spot behave like categorical features. it can be beneficial to bin these values.

- temp behave normal compared to the other features.

In [ ]:
# negative mass
df_main[df_main['mass'] < 0].shape

The negative mass issue looks like a single data point. This can be removed.

#### Feature Distribution

In [ ]:
# Building histograms
df = df_main.copy()
df.hist(figsize=(12,8), bins=30)
plt.tight_layout()
plt.show()

- This confirms what is shown by the summary statistics.

# **Feature Selection and Engineering**

This is a vital step - Feature transformation improves model performance by making data easier to learn from, especially when features have different scales or distributions. Scaling prevents features with larger numerical values from disproportionately influencing the model's learning. When features have vastly different ranges (e.g., one feature ranges from 0-10 and another from 0-10,000), the model can become biased toward the feature with the larger magnitude, simply because its values are larger. Binning and other transformations help reduce skewness, limit the impact of outliers, and capture non-linear relationships.

In [ ]:
# data points excluding negative mass
df_final = df_main[df_main['mass'] > 0]

# Remove data point of Swampp
df_final = df_final[df_final['nest'] !=  'Swampp']

In [ ]:
# create density feature from mass and volume - This feature is transformed
df_final['density'] = np.log1p(df_final['mass'].clip(lower = 0)) / np.log1p(df_final['vol'] + 1)

- This deals with the skewness of these two features.

In [ ]:
# Encoding Nest feature using target variable
le = LabelEncoder()
df_final['nest_encoded'] = le.fit_transform(df_final['nest'])

- This converts the feature to numeric because most machine learning algorithms can only work with numbers, not text or categories.

#### Data Split
Splitting data into training and testing sets is essential to evaluate a model's performance on new, unseen data and prevent overfitting. The training set teaches the model patterns, while the untouched test set provides an unbiased assessment of its generalisation ability, ensuring the model performs accurately in real-world scenarios rather than just on the data it learned from.


In [ ]:
 # Train- test Split: 80% train and 20% test -- random state = 42, ensure the reproducibility of the data split
#X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 42)

# **Model selection**

A hybrid deep learning architecture was implemented consisting of a stacked autoencoder for unsupervised feature representation learning followed by a feedforward neural network classifier. The autoencoder reduces the high-dimensional feature space into a compact latent embedding, capturing nonlinear interactions among financial variables. The classifier is composed of fully connected layers with ReLU activation, batch normalization, and dropout regularization to prevent overfitting. The model is trained in two stages: (1) unsupervised reconstruction pretraining and (2) supervised fine-tuning for binary classification using binary cross-entropy loss.

1. ****
 :  

 2. **** :

 3. **** :

# **Model Construction**

In this step, data is training data fed to the model. Training a model is essential because it allows the algorithm to learn patterns and relationships from data so it can make accurate predictions or decisions on new, unseen inputs. Without training, the model would have no understanding of how features relate to outcomes.

XGBoost contributes:
handles nonlinearity well
strong on tabular + missing patterns
captures feature interactions explicitly
 Neural Network contributes:
learns dense latent representations
captures smooth nonlinear boundaries
generalizes differently from trees
Stacking layer contributes:
learns optimal combination of both models
reduces overfitting from single model bias
improves calibration of probabilities

# **Model Evaluation**

Models are evaluated using the validation dataset. Validating these trained models is crucial because it helps identify which one performs best to unseen data—without touching the test set. This step acts like a rehearsal before the final performance: it reveals whether the model is overfitting (memorising the training data) or underfitting (failing to learn enough). By evaluating on the validation set, we can select the most promising model and fine-tune its hyperparameters, ensuring that when it is finally tested, the true predictive power is measured — not just luck or overtraining.

# **Interpretation**



In [ ]:
# Evaluation Results
#print(tabulate(results_df, headers='keys', tablefmt='grid'))